# Mitra Regressor — DIMER exported-predictor inference tutorial

**Profile:** `ARTIFACT-INFERENCE`  
**Notebook specification:** `1.0`  
**Capability:** consume an externally supplied `mitra-predictor.zip`, reconstruct the AutoGluon serving state, validate new tabular input, and produce regression predictions.

This notebook does **not** train or fine-tune a model and does not reacquire the base checkpoint from the network. The predictor ZIP must come from outside this notebook execution. It contains Python-serialized AutoGluon state, Mitra model/support context, preprocessing state, machine-readable provenance, and a digest manifest.

**By the end of this notebook you will be able to:**
- verify the uploaded archive's whole-file digest when one is supplied;
- reject unsafe archive paths, symlinks, decompression hazards, missing/unlisted files, and manifest mismatches;
- inspect model identity, immutable revision, artifact format/version, and runtime compatibility before deserialization;
- reconstruct the predictor from artifact contents alone;
- validate and score a genuinely new CSV; and
- export `predictions.csv` with source columns preserved.

**Trust boundary:** archive path checks and file digests establish integrity/consistency, not sender authenticity or safety of Python object deserialization. `TabularPredictor.load()` executes trusted serialized model state. Load only artifacts from a trusted producer and verify the whole-archive SHA-256 through a trusted channel when available.

**Data boundary:** uploaded inference data stays in the notebook runtime and is not sent to DIMER. Do not upload confidential, restricted, or sensitive data to Colab unless that environment is authorized.

**References:** [Repository README](https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/main/README.md) · [Model card](https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/main/MODEL_CARD.md) · [Producer tutorial](mitra_regressor_colab.ipynb)


## 1. Install the matching runtime and repository API

The repository's pinned tutorial requirements define the runtime. This notebook clones the repository and imports the same public `mitra_pipeline` artifact and inference validation API used by the producer tutorial.

The predictor is expected to have been exported with AutoGluon 1.5.0. Before loading serialized state, the notebook compares the artifact's recorded AutoGluon version and Python major/minor version with this runtime.

**What to look for:** Python, AutoGluon, repository commit, and the normative artifact format/version.


In [ ]:
import importlib.metadata as importlib_metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/mitra-regressor-pipeline.git"
REPO_REF = os.environ.get("DIMER_REPO_REF", "main").strip() or "main"
REPO_DIR = Path("/content/mitra-regressor-pipeline")

try:
    PREINSTALL_TORCH_VERSION = importlib_metadata.version("torch")
except importlib_metadata.PackageNotFoundError:
    PREINSTALL_TORCH_VERSION = None
TORCH_WAS_IMPORTED = "torch" in sys.modules

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

%pip install -q -r /content/mitra-regressor-pipeline/tutorials/requirements-colab.txt

INSTALLED_TORCH_VERSION = importlib_metadata.version("torch")
if TORCH_WAS_IMPORTED and INSTALLED_TORCH_VERSION != PREINSTALL_TORCH_VERSION:
    raise RuntimeError(
        "pip changed PyTorch after it had already been imported. "
        "Use Runtime → Restart session, then run the notebook top-to-bottom."
    )

sys.path.insert(0, str(REPO_DIR))
import torch
import mitra_pipeline as mp

AUTOGLUON_VERSION = importlib_metadata.version("autogluon.tabular")
print("Profile: ARTIFACT-INFERENCE")
print("Notebook spec: 1.0")
print("Python:", sys.version.split()[0])
print("AutoGluon:", AUTOGLUON_VERSION)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Repository commit:", REPO_COMMIT)
print("Artifact format:", mp.ARTIFACT_FORMAT)
print("Artifact format version:", mp.ARTIFACT_FORMAT_VERSION)


## 2. Supply and validate `mitra-predictor.zip`

Upload exactly one predictor ZIP produced by the E2E tutorial, or set `MITRA_PREDICTOR_ZIP` when executing non-interactively. The notebook validates the archive **before** calling `TabularPredictor.load()`:

- optional whole-archive SHA-256;
- no absolute, `..`, backslash, or symlink members;
- per-file, compression-ratio, and total expanded-size ceilings;
- required `artifact_manifest.json` and `tutorial_run_metadata.json`;
- exact artifact format/version, Mitra model id/revision, and regression problem type;
- every manifested file's size and SHA-256; and
- rejection of unexpected unlisted files.

The manifest proves internal consistency. It does not prove who created the archive; a malicious sender could replace both payload and manifest. Whole-archive digest verification is meaningful only when the expected digest is obtained through a trusted channel.


In [ ]:
import json

WORKDIR = Path("/content/mitra-inference")
WORKDIR.mkdir(parents=True, exist_ok=True)

EXPECTED_ZIP_SHA256 = ""  # @param {type:"string"}
supplied = os.environ.get("MITRA_PREDICTOR_ZIP", "").strip()
if supplied:
    ZIP_PATH = Path(supplied)
else:
    from google.colab import files
    uploaded = files.upload()
    zips = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith(".zip")]
    if len(zips) != 1:
        raise RuntimeError("Upload exactly one mitra-predictor.zip file.")
    name, payload = zips[0]
    ZIP_PATH = WORKDIR / Path(name).name
    ZIP_PATH.write_bytes(payload)

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Predictor ZIP not found: {ZIP_PATH}")

expected_digest = os.environ.get("MITRA_EXPECTED_ZIP_SHA256", "").strip() or EXPECTED_ZIP_SHA256.strip()
actual_digest = mp.sha256_file(ZIP_PATH)
if expected_digest:
    expected_digest = expected_digest.lower()
    if len(expected_digest) != 64 or any(ch not in "0123456789abcdef" for ch in expected_digest):
        raise ValueError("Expected ZIP SHA-256 must be a 64-character hexadecimal digest.")
    if actual_digest != expected_digest:
        raise RuntimeError(
            f"Predictor ZIP checksum mismatch. Expected {expected_digest}; got {actual_digest}."
        )
    print("✓ Whole-archive SHA-256 matches the trusted expected digest.")
else:
    print("⚠ No expected whole-archive digest supplied; integrity is limited to internal manifest checks.")

EXTRACT_ROOT = WORKDIR / "artifact"
mp.safe_extract_archive(ZIP_PATH, EXTRACT_ROOT)
artifact_manifest, run_metadata = mp.validate_artifact_directory(EXTRACT_ROOT)

print("ZIP SHA-256:", actual_digest)
print("Artifact format:", artifact_manifest["artifact_format"])
print("Artifact format version:", artifact_manifest["artifact_format_version"])
print("Base model:", run_metadata["base_model"])
print("Base revision:", run_metadata["base_model_revision"])
print("Problem type:", run_metadata["problem_type"])


## 3. Check provenance/runtime compatibility, then reconstruct serving state

The artifact's provenance is mandatory. The producer records the exact base model/revision, model-file digests, AutoGluon/Python runtime, target column, feature ordering, selected variant, and selection basis.

Because AutoGluon predictor artifacts contain Python-serialized objects, this notebook refuses to deserialize across a different AutoGluon version or Python major/minor version. That is a compatibility guard, not a security sandbox.

**What to look for:** `regression`, the pinned Mitra model/revision, the feature list, and a runtime match before the load message.


In [ ]:
artifact_python = str(run_metadata["python_version"])
artifact_python_mm = ".".join(artifact_python.split(".")[:2])
runtime_python_mm = ".".join(sys.version.split()[0].split(".")[:2])

if run_metadata["autogluon_version"] != AUTOGLUON_VERSION:
    raise RuntimeError(
        f"Artifact requires AutoGluon {run_metadata['autogluon_version']}; "
        f"runtime has {AUTOGLUON_VERSION}."
    )
if artifact_python_mm != runtime_python_mm:
    raise RuntimeError(
        f"Artifact was exported under Python {artifact_python}; "
        f"runtime is {sys.version.split()[0]}. Use matching Python major/minor."
    )

from autogluon.tabular import TabularPredictor

predictor = TabularPredictor.load(str(EXTRACT_ROOT))
if predictor.problem_type != "regression":
    raise RuntimeError(f"Expected regression predictor; loaded {predictor.problem_type!r}.")

FEATURE_COLUMNS = list(run_metadata["features"])
TARGET_COLUMN = run_metadata["target_column"]

provenance_view = {
    "AutoGluon": run_metadata["autogluon_version"],
    "Python": run_metadata["python_version"],
    "Base model": run_metadata["base_model"],
    "Base revision": run_metadata["base_model_revision"],
    "Mode": run_metadata["mode"],
    "Selection basis": run_metadata["selection_basis"],
    "Target": TARGET_COLUMN,
    "Required features": len(FEATURE_COLUMNS),
    "Data source": run_metadata.get("data_source"),
    "Producer repository commit": run_metadata.get("repository_commit"),
}
display(__import__("pandas").Series(provenance_view, name="Recorded value").to_frame())
display(__import__("pandas").DataFrame({"required_feature": FEATURE_COLUMNS}))
print("✓ Provenance/runtime compatibility established before deserialization.")
print("✓ Predictor reconstructed from artifact contents.")


## 4. Supply and validate genuinely new input

Upload one UTF-8 CSV, or set `MITRA_INFERENCE_CSV` for non-interactive execution.

**Expected schema before upload:** all required feature names shown above must be present. Column order may differ. Extra columns are allowed, are preserved in the output, and are not passed to the predictor. The target column is optional. A pre-existing `prediction` column is rejected rather than overwritten.

The repository API rejects duplicate raw headers and missing required features before inference. AutoGluon's preprocessing state is restored from the predictor artifact; it is not refitted from the uploaded rows.


In [ ]:
import pandas as pd

supplied_csv = os.environ.get("MITRA_INFERENCE_CSV", "").strip()
if supplied_csv:
    CSV_PATH = Path(supplied_csv)
    csv_name = CSV_PATH.name
    csv_payload = CSV_PATH.read_bytes()
else:
    from google.colab import files
    uploaded = files.upload()
    csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith(".csv")]
    if len(csvs) != 1:
        raise RuntimeError("Upload exactly one inference CSV.")
    csv_name, csv_payload = csvs[0]

new_data = mp.read_csv_bytes(csv_payload, csv_name)
X, extra_columns = mp.validate_inference_frame(new_data, FEATURE_COLUMNS)
if extra_columns:
    print("Extra columns preserved in predictions.csv but not passed to the model:", extra_columns)

print(f"✓ Ready for inference: {len(X):,} rows × {len(FEATURE_COLUMNS)} required features.")
display(X.head())


## 5. Predict and export `predictions.csv`

`prediction` is a scalar point estimate in the target's units. **No per-prediction uncertainty interval is provided or calibrated by this pipeline.** If a decision depends on uncertainty, evaluate error on representative labelled data and use an appropriate interval/calibration method downstream.

Machine-readable output preserves the original uploaded columns, including identifiers and extra context fields, and adds exactly one `prediction` column.


In [ ]:
predictions = mp.predict_regression(predictor, X, FEATURE_COLUMNS)

result = new_data.copy()
result["prediction"] = predictions
OUTPUT_PATH = WORKDIR / "predictions.csv"
result.to_csv(OUTPUT_PATH, index=False)

display(result.head())
print(f"✓ Wrote {len(result):,} predictions to {OUTPUT_PATH}")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(OUTPUT_PATH))


## Interpretation, limits, and troubleshooting

A successful run proves that the supplied archive passed the repository's path/size/manifest/digest checks, that its recorded model and runtime contract matched this notebook, that AutoGluon reconstructed the predictor from the artifact, that the new CSV satisfied the required feature schema, and that machine-readable regression predictions were produced.

It does **not** prove that the archive came from a trustworthy sender, that Python deserialization is safe for untrusted artifacts, or that the predictor is accurate, calibrated, fair, robust, or suitable for a production decision. Those claims require trusted artifact provenance and representative evaluation evidence.

| Failure | Meaning | Corrective action |
|---|---|---|
| whole-archive checksum mismatch | archive differs from the trusted digest | reject it and obtain the artifact again |
| unsafe path/symlink/size failure | archive violates extraction safety limits | reject it |
| missing/unlisted/digest-mismatched artifact file | archive is incomplete or altered | reject and reproduce/retransfer |
| artifact format/version mismatch | consumer and producer contracts differ | use the matching notebook/runtime |
| AutoGluon/Python mismatch | Python serialization compatibility is not established | use the recorded runtime |
| missing required features | input schema differs from the fitted predictor | add/rename the listed columns |
| existing `prediction` column | output would overwrite user data | rename or remove that column |


## AI use and provenance

This tutorial has been developed with AI assistance under human direction and review. AI attribution is authorship provenance, not independent validation. Artifact integrity checks and executed release evidence remain the basis for a specific release claim.
